# 01 Data Preparation

Inventory of the raw data (format, CRS, native resolution), creation of the anchor grid at the chosen resolution, and alignment of every layer to that grid.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from pyproj import CRS
from rasterio import features
from rasterio.enums import Resampling
from rasterio.plot import show
from rasterio.transform import from_origin
from rasterio.warp import reproject
from scipy.ndimage import distance_transform_edt
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes, mark_inset

from gis_mcdm import ask_number, crs_label, load_config, study_area_file

cfg, P = load_config()  # config.toml: study area, layers, run defaults
RAW_DIR, ALIGNED_DIR, POLYGON_DIR, FIGURES_DIR = P.raw, P.aligned, P.study_area, P.figures
for folder in (ALIGNED_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Same serif font as the figures of the other notebooks
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Palatino Linotype", "Palatino", "Book Antiqua", "TeX Gyre Pagella", "DejaVu Serif"],
})


def load_vector(path):
    gdf = gpd.read_file(path)
    return gdf if gdf.crs == TARGET_CRS else gdf.to_crs(TARGET_CRS)


def burn(gdf, out_shape, transform):
    """Rasterize all non-empty geometries as 1 on a 0 background."""
    shapes = ((g, 1) for g in gdf.geometry if g is not None and not g.is_empty)
    return features.rasterize(shapes, out_shape=out_shape, transform=transform, fill=0, dtype=np.uint8)

In [ ]:
# Run parameters (press Enter to keep the default from config.toml shown in brackets)
INTERACTIVE = cfg["run"]["interactive"]
RESOLUTION = ask_number("Which resolution (m)?", cfg["run"]["resolution"], INTERACTIVE)
RES_TAG = f"{RESOLUTION:g}m"  # appended to every output file name, e.g. "500m"

# alpha is asked only if config.toml defines gravity layers
HAS_GRAVITY = any(layer["kind"] == "gravity" for layer in cfg["layers"].values())
ALPHA = ask_number("Which alpha for the gravity layers?", cfg["run"]["gravity_alpha"], INTERACTIVE) if HAS_GRAVITY else None

## 1. Raw data inventory

In [ ]:
VECTOR_EXTS = {".shp", ".geojson", ".json", ".gpkg"}
RASTER_EXTS = {".tif", ".tiff"}
IGNORE_EXTS = {".cpg", ".dbf", ".shx", ".prj", ".sbn", ".sbx", ".qmd", ".qml",
               ".xml", ".gfs", ".pdf", ".cst", ".nc", ".py", ".txt", ".zip"}
M_PER_DEG = 111320


def describe(path):
    ext = path.suffix.lower()
    crs = res = "N/A"
    try:
        if ext in RASTER_EXTS:
            with rasterio.open(path) as src:
                crs = src.crs.to_string() if src.crs else "Unknown"
                rx, ry = src.res
                if src.crs and src.crs.is_geographic:  # degree -> metre at the raster's mean latitude
                    mid_lat = np.radians((src.bounds.top + src.bounds.bottom) / 2)
                    rx, ry = rx * M_PER_DEG * np.cos(mid_lat), ry * M_PER_DEG
                res = f"{round(rx)} x {round(ry)}"
        elif ext in VECTOR_EXTS:
            gdf = gpd.read_file(path, rows=1)  # first row is enough for the CRS
            crs = gdf.crs.to_string() if gdf.crs else "Unknown"
    except Exception:
        crs = "Error"
        if ext in RASTER_EXTS:
            res = "Error"
    return {"Layer": path.stem, "CRS": crs, "Resolution mxm": res, "Format File": ext[1:].upper()}


inventory = pd.DataFrame([describe(p) for p in RAW_DIR.rglob("*")
                          if p.is_file() and p.suffix.lower() not in IGNORE_EXTS])
if inventory.empty:
    print(f"No files were found in: {RAW_DIR}")
else:
    display(inventory)

## 2. Anchor Grid

This step creates the grid (at the user-defined RESOLUTION) that all layers are aligned to.

In [ ]:
POLYGON_PATH = study_area_file(P)
ANCHOR_PATH = POLYGON_DIR / f"anchor_grid_{RES_TAG}.tif"

# CRS of the analysis: [study_area] crs in config.toml, or the polygon's own projected CRS, or its UTM zone
study_area = gpd.read_file(POLYGON_PATH)
if cfg["study_area"].get("crs"):
    crs = CRS.from_user_input(cfg["study_area"]["crs"])
elif study_area.crs is None:
    raise ValueError(f"{POLYGON_PATH.name} has no CRS. Define it in a GIS or set [study_area] crs in config.toml.")
elif study_area.crs.is_projected and study_area.crs.axis_info[0].unit_name in ("metre", "meter"):
    crs = study_area.crs
else:
    crs = study_area.estimate_utm_crs()
    print(f"Warning: {POLYGON_PATH.name} is not in a projected CRS in metres; using {crs.name}. "
          "For a large study area, set [study_area] crs in config.toml to a national projected CRS.")
if not (crs.is_projected and crs.axis_info[0].unit_name in ("metre", "meter")):
    raise ValueError(f"The analysis CRS must be projected in metres; {crs.name} is not.")
TARGET_CRS = f"EPSG:{epsg}" if (epsg := crs.to_epsg()) else crs.to_wkt()
CRS_LABEL = crs_label(TARGET_CRS)  # for figure axes
study_area = study_area.to_crs(TARGET_CRS)
print(f"Study area: {POLYGON_PATH.name}, CRS {CRS_LABEL}")

# Snap the bounding box to multiples of RESOLUTION for clean pixel edges
minx, miny, maxx, maxy = study_area.total_bounds
minx, miny = (np.floor(v / RESOLUTION) * RESOLUTION for v in (minx, miny))
maxx, maxy = (np.ceil(v / RESOLUTION) * RESOLUTION for v in (maxx, maxy))
GRID_SHAPE = (int(round((maxy - miny) / RESOLUTION)), int(round((maxx - minx) / RESOLUTION)))  # (rows, cols)
GRID_TRANSFORM = from_origin(minx, maxy, RESOLUTION, RESOLUTION)

grid_array = burn(study_area, GRID_SHAPE, GRID_TRANSFORM)  # 1 = inside the study area, 0 = outside (NoData)
ANCHOR_MASK = grid_array == 1


def write_raster(path, array, nodata):
    profile = dict(driver="GTiff", dtype=array.dtype.name, nodata=nodata, count=1,
                   height=GRID_SHAPE[0], width=GRID_SHAPE[1], crs=TARGET_CRS, transform=GRID_TRANSFORM,
                   compress="deflate", tiled=True, blockxsize=256, blockysize=256)
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(array, 1)


write_raster(ANCHOR_PATH, grid_array, nodata=0)
print(f"Valid study-area cells: {np.count_nonzero(grid_array):,}")

In [ ]:
raster_kw = dict(transform=GRID_TRANSFORM, cmap=ListedColormap(["none", "#D0D2D3"]),
                 vmin=0, vmax=1, interpolation="none", alpha=0.9)

fig, ax = plt.subplots(figsize=(10, 10), dpi=300, facecolor="white")
show(grid_array, ax=ax, **raster_kw)
study_area.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.8)
ax.set_xlabel(f"Easting (m) - {CRS_LABEL}", fontsize=11)
ax.set_ylabel(f"Northing (m) - {CRS_LABEL}", fontsize=11)
ax.tick_params(labelsize=9)
ax.grid(True, linestyle="--", color="gray", alpha=0.3)

# Inset: 10 km x 10 km window around the centroid, with the cell lines drawn
ZOOM = 10_000
centroid = study_area.geometry.union_all().centroid
x0, y0 = centroid.x - ZOOM / 2, centroid.y - ZOOM / 2

axins = zoomed_inset_axes(ax, zoom=7, loc="lower right", borderpad=3)
show(grid_array, ax=axins, **raster_kw)
study_area.plot(ax=axins, facecolor="none", edgecolor="black", linewidth=2)
axins.set_xlim(x0, x0 + ZOOM)
axins.set_ylim(y0, y0 + ZOOM)
for v in np.arange(x0, x0 + ZOOM, RESOLUTION):
    axins.axvline(v, color="white", linewidth=0.7, alpha=0.9)
for v in np.arange(y0, y0 + ZOOM, RESOLUTION):
    axins.axhline(v, color="white", linewidth=0.7, alpha=0.9)
axins.set_xticks([])
axins.set_yticks([])
for spine in axins.spines.values():
    spine.set_edgecolor("black")
    spine.set_linewidth(1.5)
mark_inset(ax, axins, loc1=1, loc2=3, fc="none", ec="black", lw=1.2, alpha=0.8)

plt.savefig(FIGURES_DIR / f"Grid_{RES_TAG}.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Alignment to the anchor grid

In [ ]:
F_NODATA = -9999.0  # NoData of continuous layers


def source(rel_path):
    path = RAW_DIR / rel_path
    if path.exists():
        return path
    print(f"Skipping missing file: {path}")


def missing(array, nodata):
    """Boolean mask of NoData (or NaN) cells."""
    invalid = array == nodata
    if np.issubdtype(array.dtype, np.floating):
        invalid |= np.isnan(array)
    return invalid


def save(name, array, nodata):
    n_gaps = np.count_nonzero(missing(array, nodata) & ANCHOR_MASK)
    if n_gaps:
        print(f"Warning: {name} has {n_gaps:,} NoData cells inside the study area")
    array[~ANCHOR_MASK] = nodata
    write_raster(ALIGNED_DIR / f"{name}_{RES_TAG}.tif", array, nodata)
    print(f"Saved {name}_{RES_TAG}.tif")


def fill_gaps_nearest(array, nodata):
    """In place: give every NoData cell inside the study area the value of the nearest valid cell."""
    invalid = missing(array, nodata)
    gaps = invalid & ANCHOR_MASK
    if not gaps.any() or invalid.all():
        return
    nearest = np.empty((2,) + array.shape, dtype=np.int32)
    distance_transform_edt(invalid, return_distances=False, return_indices=True, indices=nearest)
    array[gaps] = array[nearest[0][gaps], nearest[1][gaps]]


def resampled(path, layer):
    """Raster resampled to the grid; gaps inside the study area (e.g. at the coast) take the nearest valid value."""
    categorical = layer.get("categorical", False)
    dtype = layer.get("dtype", "uint8" if categorical else "float32")
    nodata = layer.get("nodata", 0 if categorical else F_NODATA)
    out = np.full(GRID_SHAPE, nodata, dtype=dtype)
    with rasterio.open(path) as src:
        reproject(rasterio.band(src, 1), out, dst_transform=GRID_TRANSFORM, dst_crs=TARGET_CRS,
                  resampling=Resampling[layer.get("resampling", "mode" if categorical else "bilinear")], dst_nodata=nodata)
    fill_gaps_nearest(out, nodata)
    return out, nodata


# Cell centres, for the gravity layers
xs = GRID_TRANSFORM.c + (np.arange(GRID_SHAPE[1]) + 0.5) * GRID_TRANSFORM.a
ys = GRID_TRANSFORM.f + (np.arange(GRID_SHAPE[0]) + 0.5) * GRID_TRANSFORM.e
CHUNK = 500  # rows per block, keeps memory bounded


def gravity_surface(gdf, layer, name):
    """Sum over all features of magnitude / (distance + offset)^ALPHA; offset_m defaults to one grid cell."""
    offset = layer.get("offset_m", RESOLUTION)  # avoids division by zero
    pts = gdf.geometry.centroid  # centroids in case features are polygons
    mag_col = layer.get("magnitude")
    if mag_col in gdf.columns:
        mags = gdf[mag_col].fillna(0).to_numpy(dtype=np.float32)
    else:
        if mag_col:
            print(f"Warning: column '{mag_col}' not found in {name}; using magnitude 1.0.")
        mags = np.ones(len(gdf), dtype=np.float32)
    keep = mags > 0
    px, py, mags = pts.x.to_numpy()[keep], pts.y.to_numpy()[keep], mags[keep]

    gravity = np.zeros(GRID_SHAPE, dtype=np.float32)
    for r0 in range(0, GRID_SHAPE[0], CHUNK):
        block = gravity[r0:r0 + CHUNK]          # view into gravity
        dy = ys[r0:r0 + CHUNK, None]
        for x, y, m in zip(px, py, mags):
            dist = np.sqrt((xs - x) ** 2 + (dy - y) ** 2)  # broadcast (rows, 1) x (cols,)
            block += (m / (dist + offset) ** ALPHA).astype(np.float32)
    return gravity


# Every layer of config.toml, aligned to the anchor grid
for name, layer in cfg["layers"].items():
    kind = layer["kind"]
    out_name = f"{name}_{ALPHA:g}" if kind == "gravity" else name
    if cfg["run"]["reuse_aligned"] and (ALIGNED_DIR / f"{out_name}_{RES_TAG}.tif").exists():
        print(f"Kept {out_name}_{RES_TAG}.tif (reuse_aligned = true)")
        continue
    if not (path := source(layer["source"])):
        continue
    if kind == "raster":
        save(out_name, *resampled(path, layer))
        continue
    gdf = load_vector(path)
    if kind == "gravity":
        save(out_name, gravity_surface(gdf, layer, name), F_NODATA)
        continue
    burned = burn(gdf, GRID_SHAPE, GRID_TRANSFORM)
    if kind == "presence":  # 1 inside a feature, 0 outside, 255 outside the study area (NoData)
        save(out_name, burned, nodata=255)
    elif not burned.any():
        print(f"Skipping {name}: no features inside the grid")
    else:  # distance (m) to the nearest feature
        save(out_name, distance_transform_edt(burned == 0, sampling=RESOLUTION).astype(np.float32), F_NODATA)

print(f"Done. Aligned layers in: {ALIGNED_DIR}")